# Notebook 02 — Data Cleaning

This notebook takes the raw hourly data from `01_ingest` and produces a clean 
daily dataset ready for analysis and the Streamlit dashboard.

## Key decisions
- **Daily aggregation:** PM2.5 standards are defined as 24-hour averages. 
  Chile's legal standard (50 µg/m³) is explicitly a daily average, so 
  aggregating to daily is not just a simplification — it is the correct 
  unit of analysis.
- **Quality filter (18/24 hours):** Days with fewer than 18 valid hourly 
  readings are excluded. This follows WHO guidance and ensures daily averages 
  are representative.
- **Southern Hemisphere seasons:** December, January, and February are summer 
  (Verano) in Santiago. Santiago's worst air quality occurs in winter 
  (June–August) due to temperature inversions that trap pollution near the ground.
- **Exceedance flag:** Days where pm25_mean exceeds 50 µg/m³ are flagged with 
  `exceeds_norm = True`, based on Chile's DS 12/2011 air quality standard.

In [1]:
# Import
import pandas as pd
import numpy as np
from pathlib import Path

INTERIM_DIR = Path('../data/interim')
air = pd.read_parquet(INTERIM_DIR / 'air_quality_raw.parquet')
print(air.shape)

(531565, 4)


In [2]:
# Verify null or weird values
print(f'Values below 0   : {(air["pm25"] < 0).sum()}')
print(f'Values above 500 : {(air["pm25"] > 500).sum()}')
print(f'Missing values   : {air["pm25"].isna().sum()}')

Values below 0   : 0
Values above 500 : 0
Missing values   : 45359


In [3]:
# Comnine date and hour into a single dataframe column
air['datetime'] = air['date'] + pd.to_timedelta(air['hour'], unit='h')

print(air[['date',  'hour', 'datetime']].head(10))

        date  hour            datetime
0 2022-04-01     1 2022-04-01 01:00:00
1 2022-04-01     2 2022-04-01 02:00:00
2 2022-04-01     3 2022-04-01 03:00:00
3 2022-04-01     4 2022-04-01 04:00:00
4 2022-04-01     5 2022-04-01 05:00:00
5 2022-04-01     6 2022-04-01 06:00:00
6 2022-04-01     7 2022-04-01 07:00:00
7 2022-04-01     8 2022-04-01 08:00:00
8 2022-04-01     9 2022-04-01 09:00:00
9 2022-04-01    10 2022-04-01 10:00:00


In [4]:
# Aggregate to daily average per station
air_daily = (
    air.groupby(['station', 'date'])
    .agg(
        pm25_mean= ('pm25', 'mean'),
        pm25_max= ('pm25', 'max'),
        valid_hours= ('pm25', 'count')
    )
    .reset_index()
)

print(f'Daily records: {len(air_daily)}')
print(air_daily.head(10))

Daily records: 22149
            station       date  pm25_mean  pm25_max  valid_hours
0  Cerrillos II.csv 2022-04-01        NaN       NaN            0
1  Cerrillos II.csv 2022-04-02        NaN       NaN            0
2  Cerrillos II.csv 2022-04-03        NaN       NaN            0
3  Cerrillos II.csv 2022-04-04        NaN       NaN            0
4  Cerrillos II.csv 2022-04-05        NaN       NaN            0
5  Cerrillos II.csv 2022-04-06        NaN       NaN            0
6  Cerrillos II.csv 2022-04-07        NaN       NaN            0
7  Cerrillos II.csv 2022-04-08        NaN       NaN            0
8  Cerrillos II.csv 2022-04-09        NaN       NaN            0
9  Cerrillos II.csv 2022-04-10        NaN       NaN            0


In [5]:
print(f'Days with 0 valid hours: {(air_daily["valid_hours"] == 0).sum()}')
print(f'Days with less than 18 valid hours: {(air_daily["valid_hours"] < 18).sum()}')

Days with 0 valid hours: 1619
Days with less than 18 valid hours: 1987


In [6]:
# Apply quality filer: keep only days with at least 18 valid hours readings
air_daily = air_daily[air_daily['valid_hours'] >=18].copy()

print(f'Daily records after quality filter: {len(air_daily)}')
print(f'Records removed: {1987}')

Daily records after quality filter: 20162
Records removed: 1987


In [7]:
# Add temporal features
air_daily['year'] = air_daily['date'].dt.year
air_daily['month'] = air_daily['date'].dt.month

#Southern Hemisphere seasons
def get_season(month):
    if month in [12, 1, 2]:
        return 'Summer'
    elif month in [3, 4, 5]:
        return 'Autumn'
    elif month in [6, 7, 8]:
        return 'Winter'
    else:
        return 'Spring'

air_daily['season'] = air_daily['month'].apply(get_season)

print(air_daily[['date', 'month', 'season']]. head(10))
print(air_daily['season'].value_counts())

         date  month  season
11 2022-04-12      4  Autumn
12 2022-04-13      4  Autumn
13 2022-04-14      4  Autumn
14 2022-04-15      4  Autumn
15 2022-04-16      4  Autumn
16 2022-04-17      4  Autumn
17 2022-04-18      4  Autumn
18 2022-04-19      4  Autumn
19 2022-04-20      4  Autumn
20 2022-04-21      4  Autumn
season
Winter    5209
Spring    5053
Summer    5025
Autumn    4875
Name: count, dtype: int64


In [8]:
# Flag days exceeding Chilean PM2.5 norm (50 μg/m³)
air_daily['exceeds_norm'] = air_daily['pm25_mean'] > 50

print(f'Days exceeding norm: {air_daily["exceeds_norm"].sum()}')
print(f'Percentage: {air_daily["exceeds_norm"].mean()*100:.1f}%')

Days exceeding norm: 1833
Percentage: 9.1%


In [13]:
# Fix stations name
air_daily['station'] = air_daily['station'].str.replace('.csv', '', regex=False)
print(air_daily['station'].unique())

<ArrowStringArray>
[    'Cerrillos II',      'Cerro Navia',        'El Bosque',
    'Independencia',       'La Florida',       'Las Condes',
 'Parque O'Higgins',         'Pudahuel',      'Puente Alto',
        'Quilicura',        'Talagante']
Length: 11, dtype: str


## Cleaning summary

- Started with 531,565 hourly records across 11 stations.
- Aggregated to daily averages, producing 22,149 station-day combinations.
- Quality filter (minimum 18 valid hourly readings): removed 1,987 days, 
  leaving 20,162 daily records.
- 9.1% of days (1,833) exceed Chile's PM2.5 legal standard of 50 µg/m³.
- Southern Hemisphere seasons applied correctly: winter peaks expected in 
  June–August due to temperature inversions trapping pollution in Santiago's 
  basin geography.

In [14]:
# Summary
print(f'Total daily recods  : {len(air_daily)}')
print(f'Stations            : {air_daily["station"].nunique()}')
print(f'Date range          : {air_daily["date"].min()} to {air_daily["date"].max()}')
print(f'Missing pm25_mean   : {air_daily["pm25_mean"].isna().sum()}')
print(f'Days exceeding norm : {air_daily["exceeds_norm"].sum()} ({air_daily["exceeds_norm"].mean()*100:.1f}%)')
print(f'Columns             : {air_daily.columns.tolist()}')

Total daily recods  : 20162
Stations            : 11
Date range          : 2020-01-01 00:00:00 to 2025-12-31 00:00:00
Missing pm25_mean   : 0
Days exceeding norm : 1833 (9.1%)
Columns             : ['station', 'date', 'pm25_mean', 'pm25_max', 'valid_hours', 'year', 'month', 'season', 'exceeds_norm']


In [15]:
# Save
air_daily.to_parquet(INTERIM_DIR / 'air_quality_daily.parquet', index=False)
print(f'Saved successfully')

Saved successfully
